# 1 · Juros altos se justificam?

**Pergunta:** o patamar de juros contemporâneo é explicado por inflação, atividade
e emprego — ou há um componente institucional a mais do que essas variáveis pedem?

**O que argumentar de forma defensável exige mostrar, com dados:**
1. que o ciclo da Selic acompanha (com defasagem) a inflação realizada **e esperada** (Focus);
2. o histórico e o patamar atual do **juro real ex ante** (Selic − inflação esperada), e como
   ele se compara à média histórica;
3. se atividade (IBC-Br) e desemprego explicam a direção;
4. onde a Selic fica em relação a uma **regra de Taylor** simples — referência explícita de
   "o quanto é mais do que uma regra básica pediria" (não é "o juro correto").

**Fontes:** SGS 432/4189/13522/24364; SIDRA 6381 v4099 (desemprego, trimestre móvel);
Focus/Olinda (`ExpectativasMercadoInflacao12Meses`, média não suavizada).


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
m = ind.selic_real_mensal()
m = m.drop(columns=["selic_mensal"])
print(m.tail(3).round(2).to_string())
print(f"\nperíodo: {m.index.min().date()} → {m.index.max().date()}")

            selic_meta  ipca_12m  focus_ipca_12m  ibcbr_var12m  desemprego  juro_real_ex_post  juro_real_ex_ante  juro_real_ex_post_fisher  juro_real_ex_ante_fisher  juro_real_ex_post_12m
2026-09-30       13.75      4.58            4.81          0.86        5.20               9.20               8.97                      8.80                      8.56                   9.96
2026-10-31       13.75      4.63            4.90           NaN         NaN               9.02               8.75                      8.62                      8.34                   9.81
2026-11-30       13.75       NaN            5.15           NaN         NaN                NaN               8.50                       NaN                      8.08                    NaN

período: 2002-01-31 → 2026-11-30


In [3]:
fig, ax = plt.subplots(figsize=(11, 5))
m["selic_meta"].plot(ax=ax, color=AZUL, label="Selic meta (fim de mês)")
m["ipca_12m"].dropna().plot(ax=ax, color=VERMELHO, label="IPCA 12m (realizado)")
m["focus_ipca_12m"].dropna().plot(ax=ax, color=LARANJA, ls="--", label="Focus: IPCA 12m (média)")
ind.meta_inflacao().plot(ax=ax, color=CINZA, lw=1, ls=":",
                         label="meta de inflação (4,5% até 2024; 3,0% desde 2025)")
ax.set_ylabel("% a.a.")
ax.set_title("Selic meta vs inflação realizada e esperada (2002–2026)")
ax.legend()
salvar(fig, "01_selic_vs_inflacao")

fig -> viz/figs/01_selic_vs_inflacao.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_selic_vs_inflacao.png')

In [4]:
fig, ax = plt.subplots(figsize=(11, 5))
m["juro_real_ex_ante"].dropna().plot(ax=ax, color=VERDE, label="juro real ex ante (Selic − Focus)")
m["juro_real_ex_post"].dropna().plot(ax=ax, color=CINZA, label="juro real ex post (Selic − IPCA)")
ax.axhline(0, color="k", lw=1)
media_ant = m.loc[: "2014", "juro_real_ex_ante"].mean()
ax.axhline(media_ant, color=VERDE, ls=":", lw=1.2)
ax.text(pd.Timestamp("2002-06-01"), media_ant + 0.4, f"média 2002–2014: {media_ant:.1f}%", fontsize=9, color=VERDE)
ax.set_ylabel("% a.a.")
ax.set_title("Juro real brasileiro: ex ante vs ex post")
ax.legend()
salvar(fig, "01_juro_real")

fig -> viz/figs/01_juro_real.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_juro_real.png')

In [5]:
janelas = [
    ("2002-2014", slice("2002", "2014")),
    ("2015-2019", slice("2015", "2019")),
    ("2020-2021 (juros a 2%)", slice("2020", "2021")),
    ("2022-2023 (aperto pós-pandemia)", slice("2022", "2023")),
    ("2024-2026 (patamar atual)", slice("2024", "2026")),
]
linhas = []
for rotulo, jan in janelas:
    rec = m.loc[jan]
    linhas.append({
        "período": rotulo,
        "selic média": rec.selic_meta.mean(),
        "juro real ex ante médio": rec.juro_real_ex_ante.mean(),
        "desemprego médio": rec.desemprego.mean(),
    })
est = pd.DataFrame(linhas).set_index("período")
print(est.round(1).to_string())

                                 selic média  juro real ex ante médio  desemprego médio
período                                                                                
2002-2014                              13.60                     8.10              7.10
2015-2019                              10.00                     5.10             11.40
2020-2021 (juros a 2%)                  3.80                    -0.40             13.50
2022-2023 (aperto pós-pandemia)        12.90                     7.80              8.80
2024-2026 (patamar atual)              13.20                     8.80              6.20


In [6]:
fig, ax = plt.subplots(figsize=(11, 5))
ax2 = ax.twinx()
m["desemprego"].dropna().plot(ax=ax, color=ROXO, label="desemprego (3m móvel, eixo esq.)")
m["ibcbr_var12m"].dropna().plot(ax=ax2, color=AZUL, label="IBC-Br var 12m % (eixo dir.)")
m["juro_real_ex_ante"].dropna().plot(ax=ax, color=VERMELHO, alpha=0.45, label="juro real ex ante")
ax2.set_ylim(-15, 15)
ax.set_ylabel("%")
ax2.set_ylabel("%")
ax.set_title("Juro real vs atividade e emprego")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
salvar(fig, "01_juro_real_vs_atividade")

fig -> viz/figs/01_juro_real_vs_atividade.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_juro_real_vs_atividade.png')

In [7]:
# Regra de Taylor simples (r* = 2,5%; meta VARIÁVEL; peso 0,5) — referência,
# não "juro correto". Meta: 4,5% até 2024, 3,0% desde 2025 (BCB); com a meta
# fixa em 4,5, o Taylor de 2025–26 fica 0,75 p.p. abaixo e o desvio infla.
t = m[["selic_meta", "focus_ipca_12m"]].dropna()
meta = ind.meta_inflacao().reindex(t.index).ffill()
t["taylor"] = 2.5 + t["focus_ipca_12m"] + 0.5 * (t["focus_ipca_12m"] - meta)
t["desvio"] = t["selic_meta"] - t["taylor"]
# r* é PREMISSA, não dado: o desvio cai 1:1 com r*. A faixa abaixo cobre o
# r*=2,5 (base) até r*=5; a entrevista verificada de Kanczuk (25/07/2025)
# discute r* de 8% (ele), 5% (BC) e 6% (consenso) — ver documentos.py.
t["desvio_r5"] = t["selic_meta"] - (5 + t["focus_ipca_12m"] + 0.5 * (t["focus_ipca_12m"] - meta))

print("Selic vs regra de Taylor (fim de ano):")
print(t.resample("YE").last().round(1).to_string())
print("Desvio médio (Selic − Taylor) por período:")
for nome, jan in [("2003-14", slice("2003", "2014")), ("2015-19", slice("2015", "2019")),
                  ("2019-21", slice("2019", "2021")), ("2022-23", slice("2022", "2023")),
                  ("2024-26", slice("2024", "2026"))]:
    print(f"  {nome}: {t.loc[jan, 'desvio'].mean():.1f} p.p.")
print("Sensibilidade a r* (desvio médio 2024–26):")
for r in (2.5, 4, 5):
    tv = r + t["focus_ipca_12m"] + 0.5 * (t["focus_ipca_12m"] - meta)
    print(f"  r*={r}: {(t['selic_meta'] - tv)[t.index.year >= 2024].mean():.1f} p.p.")

Selic vs regra de Taylor (fim de ano):
            selic_meta  focus_ipca_12m  taylor  desvio  desvio_r5
2002-12-31       25.00           13.20   20.10    4.90       2.40
2003-12-31       16.50            5.90    9.10    7.40       4.90
2004-12-31       17.80            6.10    9.40    8.40       5.90
2005-12-31       18.00            4.50    6.90   11.10       8.60
2006-12-31       13.20            4.10    6.40    6.90       4.40
2007-12-31       11.20            4.40    6.90    4.30       1.80
2008-12-31       13.80            4.90    7.60    6.10       3.60
2009-12-31        8.80            4.40    6.80    1.90      -0.60
2010-12-31       10.80            5.50    8.50    2.20      -0.30
2011-12-31       11.00            5.40    8.30    2.70       0.20
2012-12-31        7.20            5.60    8.70   -1.40      -3.90
2013-12-31       10.00            6.10    9.40    0.60      -1.90
2014-12-31       11.80            6.70   10.30    1.40      -1.10
2015-12-31       14.20            7.2

In [8]:
fig, ax = plt.subplots(figsize=(11, 5))
t["selic_meta"].plot(ax=ax, color=AZUL, label="Selic meta")
t["taylor"].plot(ax=ax, color=CINZA, ls="--", label="Regra de Taylor simples (r*=2,5; meta 4,5→3,0)")
t["desvio"].plot(ax=ax, color=VERMELHO, alpha=0.4, label="desvio (Selic − Taylor, r*=2,5)")
ax.fill_between(t.index, t["desvio_r5"], t["desvio"], color=VERMELHO, alpha=0.15,
                label="sensibilidade do desvio (r* de 2,5 a 5)")
ax.axhline(t["desvio"].mean(), color=VERMELHO, ls=":", lw=1)
ax.set_ylabel("% a.a.")
ax.set_title("O que a regra de Taylor pediria e onde a Selic ficou")
ax.legend(fontsize=9)
salvar(fig, "01_taylor")

fig -> viz/figs/01_taylor.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_taylor.png')

**Achados verificáveis (P1).**

- A Selic acompanha a inflação **esperada** (Focus) com defasagem de poucos
  meses — a *direção* do ciclo não é arbitrária. O que se destaca é a
  **magnitude**: com Focus 12m em torno de 5% (5,15% em nov/26) e Selic em 13–15% a.a., o juro real
  ex ante de 2024–2026 tem **média de ~8,8 p.p.** (6,9 em 2024 → 9,6 em 2025
  → 9,9 em 2026) e **pico de 11,0 p.p. em fev/2026** — a média mais alta
  desde 2016. Níveis do mesmo porte já ocorreram em outras situações:
  inflação em dois dígitos (2003–04), ajuste pós-choque (2015–16), picos de
  crise (2008) e o aperto pós-pandemia (2022–23). E hoje ele coexiste com
  inflação de ~4,6% — acima do teto de 4,5% da meta de 3% (em vigor
  desde 2025). Não é a primeira vez: o IPCA furou o teto em 2015, 2021,
  2022 e 2024 (e em 2026, ainda incompleto); ficou dentro em 2012–14,
  2016, 2018–20, 2023 e 2025, e abaixo do piso em 2017.
- A regra de Taylor simples (meta de 3% desde 2025) pediria Selic de
  ~8,2–8,8% a.a. para inflação esperada de ~5%. O **desvio médio atual
  (2024–26: ~6,0 p.p.) é o maior entre as janelas comparadas** (2003–14:
  ~4,9; 2022–23: ~5,2 p.p.) — mas
  **não é recorde histórico**: a MM36 chegou a 9,9 p.p. em fev/2006. Nos
  dois primeiros mandatos Lula o
  excesso tinha contrapeso: Focus de 10–14% (2003) e risco soberano. Hoje,
  com expectativa de ~5% e meta de 3%, o desvio não tem contrapeso
  inflacionário equivalente. O desvio é sensível à premissa r*: ~6,0 p.p.
  com r*=2,5, ~4,5 com r*=4 e ~3,5 com r*=5 — a entrevista verificada de
  Kanczuk (25/07/2025) discute r* de 8% (ele), 5% (BC) e 6% (consenso);
  com r*=8, o prêmio se anularia. O número certo é uma faixa, não um ponto.
- Atividade e emprego não explicam o patamar: com IBC-Br de +2–3% e
  desemprego em mínimos (5,2–5,3%), a justificativa de "esfriar economia
  superaquecida" fica restrita — o aperto persiste em função de juízo
  institucional sobre credibilidade, não de variáveis observadas.

**O que isso permite concluir sem depender de documentos/declarações:** o
juro extra atual não é "preço da inflação": é um **prêmio institucional
adicional**, de magnitude sem precedente recente mesmo com inflação em
meta — quem defende que "está justificado" precisa apontar a causa
adicional (risco fiscal? credibilidade?) e ela não está nos dados
macro aqui usados.

## O que o mercado projetou e o que aconteceu

O Focus é uma pesquisa **do mercado** — bancos, gestoras e consultorias —,
não do Banco Central. Esta seção compara o que ele projetava com o que de
fato aconteceu, em duas réguas:

- **anos** para as metas anuais (IPCA do ano, Selic meta no fim do ano, PIB
  real e dívida bruta % do PIB): a expectativa é a **primeira pesquisa de
  janeiro** do próprio ano (e a de julho, para ver a convergência), contra o
  realizado em dezembro — IPCA 12m (SGS 13522), Selic meta (SGS 432), dívida
  bruta (SGS 13762) e PIB real (IBGE/SIDRA);
- **trimestres** para a expectativa de **12 meses** (Focus 12m, média do
  trimestre), contra o IPCA 12m realizado **12 meses depois**. Semestre não
  acrescenta: a série de 12 meses atravessa anos, e o trimestre dá 95 pontos
  para medir o erro.

Fonte: Focus (focus_anuais, focus_inflacao12m), SGS e IBGE/SIDRA.


In [9]:
from src.extratores.base import ler_parquet
foc = ind.focus_anuais()
mm = ind.macro_mensal()
pib = ind.crescimento_pib_real()
sg = ler_parquet("sgs")

def dez(nome, ano):
    s = sg[sg.serie == nome].sort_values("data")
    d = s[(s.data.dt.year == ano) & (s.data.dt.month == 12)]
    return float(d.valor.iloc[-1])

def vintage(indicador, ref, mes):
    d = foc[(foc.indicador == indicador) & (foc.data_referencia == ref) &
            (foc.data_pesquisa.dt.year == ref) &
            (foc.data_pesquisa.dt.month == mes)]
    return float(d.sort_values("data_pesquisa").media.iloc[0])

def tabela(indicador, realizado, unidade):
    linhas = []
    for ano in range(2018, 2026):
        jan = vintage(indicador, ano, 1)
        jul = vintage(indicador, ano, 7)
        r = realizado(ano)
        linhas.append({"ano": ano, "jan": jan, "jul": jul,
                       "realizado": r, "erro (jan)": r - jan})
    t = pd.DataFrame(linhas).set_index("ano").round(2)
    return t.rename(columns=lambda c: f"{c} ({unidade})")

print("IPCA do ano — Focus (média) vs realizado em dezembro (%)")
print(tabela("IPCA", lambda a: dez("ipca_12m", a), "%").to_string(), "\n")
print("Selic meta no fim do ano — Focus vs realizado (% a.a.)")
print(tabela("Selic", lambda a: dez("meta_selic_copom", a), "% a.a.").to_string(), "\n")
print("PIB real do ano — Focus vs realizado (IBGE) (%)")
print(tabela("PIB Total", lambda a: float(pib.loc[a]), "%").to_string(), "\n")
print("Dívida bruta do governo geral — Focus vs realizado (% do PIB)")
print(tabela("Dívida bruta do governo geral", lambda a: dez("dbgg_pib", a),
             "% do PIB").to_string())


IPCA do ano — Focus (média) vs realizado em dezembro (%)
      jan (%)  jul (%)  realizado (%)  erro (jan) (%)
ano                                                  
2018     3.98     4.08           3.75           -0.23
2019     4.04     3.81           4.31            0.27
2020     3.59     1.64           4.52            0.93
2021     3.32     6.04          10.06            6.74
2022     5.10     8.05           5.79            0.69
2023     5.32     5.03           4.62           -0.70
2024     3.87     4.04           4.83            0.96
2025     4.99     5.20           4.26           -0.73 

Selic meta no fim do ano — Focus vs realizado (% a.a.)
      jan (% a.a.)  jul (% a.a.)  realizado (% a.a.)  erro (jan) (% a.a.)
ano                                                                      
2018          6.89          6.62                6.50                -0.39
2019          7.16          5.65                4.50                -2.66
2020          4.48          2.00                2.

      jan (%)  jul (%)  realizado (%)  erro (jan) (%)
ano                                                  
2018     2.64     1.62           1.78           -0.86
2019     2.59     0.88           1.22           -1.37
2020     2.31    -6.55          -3.28           -5.59
2021     3.47     5.07           4.76            1.29
2022     0.29     1.49           3.02            2.72
2023     0.80     2.19           3.24            2.44
2024     1.56     2.08           3.42            1.86
2025     2.04     2.25           2.29            0.24 

Dívida bruta do governo geral — Focus vs realizado (% do PIB)


      jan (% do PIB)  jul (% do PIB)  realizado (% do PIB)  erro (jan) (% do PIB)
ano                                                                              
2018           77.77           76.95                 75.27                  -2.50
2019           79.00           79.15                 74.44                  -4.56
2020           78.86           92.05                 86.94                   8.08
2021           92.14           84.03                 77.31                 -14.83
2022           84.19           79.13                 71.68                 -12.51
2023           80.09           76.98                 73.83                  -6.26
2024           78.90           78.03                 76.27                  -2.63
2025           82.20           80.39                 78.64                  -3.56


In [10]:
exp = mm["focus_ipca_12m"].dropna()
real12 = mm["ipca_12m"].dropna()
linhas = []
for t in pd.period_range("2002Q1", "2025Q4", freq="Q"):
    e = exp[(exp.index >= t.start_time) & (exp.index <= t.end_time)]
    alvo = (t.asfreq("M") + 12).to_timestamp(how="end").normalize()
    if e.empty or alvo not in real12.index:
        continue
    linhas.append({"trimestre": str(t), "expectativa": e.mean(),
                   "realizado": float(real12.loc[alvo])})
t12 = pd.DataFrame(linhas)
t12["erro"] = t12.realizado - t12.expectativa
print(f"{len(t12)} trimestres | erro médio {t12.erro.mean():+.2f} p.p. | "
      f"erro absoluto médio {t12.erro.abs().mean():.2f} p.p. | "
      f"maior subestimação {t12.erro.max():+.2f} p.p. | "
      f"maior superestimação {t12.erro.min():+.2f} p.p.")
print()
print("trimestres recentes (2019 em diante):")
print(t12[t12.trimestre >= "2019Q1"].round(2).to_string(index=False))


95 trimestres | erro médio +0.90 p.p. | erro absoluto médio 1.86 p.p. | maior subestimação +11.96 p.p. | maior superestimação -5.01 p.p.

trimestres recentes (2019 em diante):
trimestre  expectativa  realizado  erro
   2019Q1         3.97       3.30 -0.67
   2019Q2         3.58       2.13 -1.45
   2019Q3         3.49       3.14 -0.35
   2019Q4         3.76       4.52  0.76
   2020Q1         3.34       6.10  2.76
   2020Q2         2.80       8.35  5.55
   2020Q3         3.24      10.25  7.01
   2020Q4         4.04      10.06  6.02
   2021Q1         4.02      11.30  7.28
   2021Q2         4.34      11.89  7.55
   2021Q3         5.02       7.17  2.15
   2021Q4         5.54       5.79  0.25
   2022Q1         5.71       4.65 -1.06
   2022Q2         5.98       3.16 -2.82
   2022Q3         4.89       5.19  0.30
   2022Q4         5.26       4.62 -0.64
   2023Q1         5.68       3.93 -1.75
   2023Q2         4.66       4.23 -0.43
   2023Q3         4.14       4.42  0.28
   2023Q4         3.88  

In [11]:
fig, ax = plt.subplots(figsize=(11, 5))
x = pd.PeriodIndex(t12.trimestre, freq="Q").to_timestamp(how="end")
ax.plot(x, t12.expectativa, color=LARANJA, lw=1.6, ls="--",
        label="Focus: IPCA para os próximos 12 meses (média do trimestre)")
ax.plot(x, t12.realizado, color=VERMELHO, lw=1.6,
        label="IPCA 12m realizado 12 meses depois")
ax.set_ylabel("% a.a.")
ax.set_title("O que o mercado esperava e o que aconteceu — IPCA 12 meses (2002–2025)")
ax.legend(fontsize=9, frameon=False)
salvar(fig, "01_focus_vs_realizado")


fig -> viz/figs/01_focus_vs_realizado.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_focus_vs_realizado.png')

**O que a comparação mostra:**

- **O Focus erra grande nos anos de choque, nas duas direções.** Na
  projeção anual, em janeiro de 2021 ele projetava IPCA de 3,3% para 2021 —
  o realizado em dezembro foi 10,1% (+6,7 p.p.); a Selic meta, 3,3% → 9,25%
  (+6,0 p.p.). Na série de 12 meses, a expectativa média do 1º tri/2021 era
  4,0% e o realizado 12 meses depois foi 11,3% (+7,3 p.p.). Em 2020, PIB de
  +2,3% → −3,3% (−5,6 p.p.) e dívida de 78,9% → 86,9% do PIB.
- **No ciclo de juros altos (2022–2025), o erro de inflação encolheu**
  (média anual entre −1,1 e +1,1 p.p.), mas o **PIB foi sistematicamente
  subestimado**: 2022 projetado 0,3% vs 3,0% realizado; 2023, 0,8 vs 3,2;
  2024, 1,6 vs 3,4. O mercado foi pessimista com o crescimento.
- **A dívida bruta surpreendeu para baixo em 2021–2025** (projetava 92,1% em
  2021; o realizado foi 77,3%): a inflação e o PIB nominal altos diluíram a
  dívida. O "enxugando gelo" **não** aparece como dívida explodindo no
  realizado — aparece como **transferência**: ~R$ 1 tri/ano de juros pagos
  com a dívida caindo em % do PIB.
- **A expectativa de 12 meses tem erro absoluto médio de 1,9 p.p.** em 95
  trimestres: é um termômetro do que o mercado espera — com erros grandes nos
  pontos de virada —, não uma validação independente da política de juros.


In [12]:
ult = foc.data_pesquisa.max()
print(f"o que o mercado espera hoje (pesquisa de {ult:%d/%m/%Y}):")
for ref in (2026, 2027, 2028):
    def valor(indicador):
        d = foc[(foc.indicador == indicador) & (foc.data_referencia == ref) &
                (foc.data_pesquisa == ult)]
        return float(d.media.iloc[0])
    s, pi, g = valor("Selic"), valor("IPCA"), valor("PIB Total")
    print(f"  {ref}: Selic {s:.2f}% | IPCA {pi:.2f}% | PIB {g:.2f}% "
          f"→ juro real esperado {s - pi:.2f}% | i-g {s - pi - g:.2f} p.p.")


o que o mercado espera hoje (pesquisa de 13/11/2026):
  2026: Selic 13.68% | IPCA 5.11% | PIB 1.73% → juro real esperado 8.57% | i-g 6.84 p.p.
  2027: Selic 12.78% | IPCA 4.96% | PIB 1.05% → juro real esperado 7.82% | i-g 6.77 p.p.
  2028: Selic 11.36% | IPCA 4.05% | PIB 1.72% → juro real esperado 7.31% | i-g 5.58 p.p.
